In [1]:
import polars as pl
import src.curvas_financiacion as cfin

# Lee insumos
input_infl = pl.read_excel('./inputs/inflacion_mensual.xlsx')
df_ipc = cfin.procesar_inflacion(input_infl)
df_ipc

mesid_ipc,tasa,indice_ipc
i32,f64,f64
202104,0.005923,1.005923
202105,0.0099945,1.015977
202106,-0.000491,1.015477
202107,0.0032499,1.018778
202108,0.0044742,1.023336
…,…,…
202508,0.0018768,1.40953
202509,0.003196,1.414035
202510,0.001842,1.416639


In [2]:
input_curvas = pl.read_excel('./inputs/irr_mensual_202411.xlsx')
input_curvas

mes,pais,consecutivo,moneda,fecha_clave,tasa_interes
i64,str,str,str,i64,f64
1,"""CO""","""CO_COP001M""","""COP""",20241130,9.111264
2,"""CO""","""CO_COP002M""","""COP""",20241130,9.111264
3,"""CO""","""CO_COP003M""","""COP""",20241130,9.111264
4,"""CO""","""CO_COP004M""","""COP""",20241130,9.111264
5,"""CO""","""CO_COP005M""","""COP""",20241130,9.111264
…,…,…,…,…,…
1316,"""CO""","""CO_USD1316M""","""USD""",20241130,5.2714
1317,"""CO""","""CO_USD1317M""","""USD""",20241130,5.2714
1318,"""CO""","""CO_USD1318M""","""USD""",20241130,5.2714


In [3]:
param_compfin = pl.read_excel('./inputs/insumos.xlsx', sheet_name='componente_financiacion')
param_compfin

tipo_contabilidad,tipo_insumo,compania,ramo_sura,producto,tipo_op,moneda,aplica_comp_financ,aplica_ipc_mensual,pais_curva,moneda_curva,meses_max_vigencia
str,str,str,str,str,str,str,i64,i64,str,str,i64
"""ifrs17_local""","""*""","""01""","""003""","""*""","""*""","""COP""",1,1,"""CO""","""UVR""",120
"""ifrs17_corporativo""","""*""","""01""","""003""","""*""","""*""","""COP""",1,1,"""CO""","""UVR""",120


In [4]:
# Procesa las curvas para derivar factores financieros
df_factores_interes = cfin.procesar_curvas_tasas(input_curvas, param_compfin)
df_factores_interes

mesid_curva,fecha_curva,moneda_curva,pais_curva,nodo,mesid_valoracion,fecha_valoracion,tasa_efectiva_anual,tasa_fwd_real,factor_acumulacion,factor_desc_real,sum_desc_real
i32,date,str,str,i64,i32,date,f64,f64,f64,f64,f64
202411,2024-11-30,"""UVR""","""CO""",1,202412,2024-12-31,0.035113,0.00288,1.00288,0.997128,0.997128
202411,2024-11-30,"""UVR""","""CO""",2,202501,2025-01-31,0.035113,0.00288,1.005768,0.994265,1.991393
202411,2024-11-30,"""UVR""","""CO""",3,202502,2025-02-28,0.035113,0.00288,1.008665,0.991409,2.982802
202411,2024-11-30,"""UVR""","""CO""",4,202503,2025-03-31,0.035113,0.00288,1.01157,0.988562,3.971365
202411,2024-11-30,"""UVR""","""CO""",5,202504,2025-04-30,0.035113,0.00288,1.014483,0.985723,4.957088
…,…,…,…,…,…,…,…,…,…,…,…
202411,2024-11-30,"""UVR""","""CO""",118,203409,2034-09-30,0.052358,0.004138,1.651734,0.605425,92.979759
202411,2024-11-30,"""UVR""","""CO""",119,203410,2034-10-31,0.052344,0.004132,1.658559,0.602933,93.582693
202411,2024-11-30,"""UVR""","""CO""",120,203411,2034-11-30,0.052331,0.004134,1.665415,0.600451,94.183143


# Prueba Pipeline

In [5]:
import src.parametros as p
import src.aux_tools as aux_tools
import src.prep_insumo as prep_data
import src.devenga as devg
import src.fluctuacion as fluc
import src.mapeo_contable as mapcont
import polars as pl


# lee los parámetros e insumos relevantes
FECHA_VALORACION = p.FECHA_VALORACION
FECHA_VALORACION_ANTERIOR = p.FECHA_VALORACION_ANTERIOR
FECHA_TRANSICION = p.FECHA_TRANSICION
# Insumos transversales
param_contab = pl.read_excel(
    p.RUTA_INSUMOS, sheet_name=p.HOJA_PARAMETROS_CONTAB
).filter(pl.col("estado_insumo") == 1)  # Solo se usan las configuraciones activas (1)
excepciones = pl.read_excel(p.RUTA_INSUMOS, sheet_name=p.HOJA_EXCEPCIONES_50_50)
gasto = pl.read_excel(p.RUTA_GASTOS)
tasa_cambio = pl.read_excel(p.RUTA_INSUMOS, sheet_name=p.HOJA_MONEDA)
descuentos = pl.read_excel(p.RUTA_INSUMOS, sheet_name=p.HOJA_DESCUENTO)
# El diccionario o tabla de correspondencia de outputs con entradas contables
input_map_bts = pl.read_excel(p.RUTA_REL_BT)
input_tipo_seguro = pl.read_excel(p.RUTA_INSUMOS, sheet_name=p.HOJA_TIPO_SEGURO)
# Insumos de cuenta corriente y cartera
# input_cuentas_reaseguro = pl.read_excel(p.RUTA_INSUMOS, sheet_name=p.HOJA_CUENTAS_REA)
# Insumo de riesgo de credito reaseguradores
input_riesgo_credito = pl.read_excel(p.RUTA_RIESGO_CREDITO)

# Insumos de recibos contabilizados en SAP
produccion_dir = pl.read_excel(p.RUTA_INSUMOS, sheet_name=p.HOJA_PDN)
cesion_rea = pl.read_excel(p.RUTA_INSUMOS, sheet_name=p.HOJA_CESION)

# prepara el insumo para el devengo de produccion directo
input_prod_dir = prep_data.prep_input_prima_directo(
    produccion_dir, param_contab, excepciones, FECHA_VALORACION
)
input_prod_dir

Could not determine dtype for column 1, falling back to string
Could not determine dtype for column 4, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 4, falling back to string
Could not determine dtype for column 15, falling back to string
Could not determine dtype for column 16, falling back to string


aux_casuistica,tipo_insumo,tipo_negocio,poliza,fecha_expedicion_poliza,recibo,numero_documento_sap,amparo,cdsubgarantia,poliza_certificado,compania,ramo_sura,canal,producto,tipo_op,moneda,fecha_contabilizacion_recibo,fecha_inicio_vigencia_recibo,fecha_fin_vigencia_recibo,fecha_inicio_vigencia_cobertura,fecha_fin_vigencia_cobertura,valor_prima_emitida,tipo_contabilidad,componente,clasificacion_adicional,tipo_contrato,nivel_detalle,signo_constitucion,candidato_devengo_50_50,fecha_inicio_vigencia,fecha_constitucion,fecha_inicio_devengo,fecha_fin_devengo,valor_base_devengo
str,str,str,str,date,str,i64,str,str,str,str,str,str,str,str,str,date,date,date,date,date,f64,str,str,str,str,str,i64,i32,date,date,date,date,f64
"""garantia_extendida_pre_riesgo""","""produccion_directo""","""directo""","""259823906""",2023-07-15,"""2940977660""",2940977660,"""DANOS""","""DANOS""","""239524652""","""01""","""009""","""sucursal""","""plan_3""","""emision""","""COP""",2023-07-15,2023-01-15,2025-01-15,2023-01-15,2025-01-15,1200.0,"""ifrs17_corporativo""","""prima""","""reserva""","""directo""","""cobertura""",-1,0,2023-01-15,2023-07-15,2023-07-15,2025-01-15,1200.0
"""garantia_extendida_pre_riesgo""","""produccion_directo""","""directo""","""259823906""",2023-07-15,"""2940977660""",2940977660,"""DANOS""","""DANOS""","""239524652""","""01""","""009""","""sucursal""","""plan_3""","""emision""","""COP""",2023-07-15,2023-01-15,2025-01-15,2023-01-15,2025-01-15,1200.0,"""ifrs17_local""","""prima""","""reserva""","""directo""","""cobertura""",-1,0,2023-01-15,2023-07-15,2023-07-15,2025-01-15,1200.0
"""garantia_extendida_pre_riesgo""","""produccion_directo""","""directo""","""259823906""",2023-07-15,"""2940977660""",2940977660,"""DANOS""","""DANOS""","""239524652""","""01""","""009""","""sucursal""","""plan_3""","""emision""","""COP""",2023-07-15,2023-01-15,2025-01-15,2023-01-15,2025-01-15,1200.0,"""ifrs4""","""prima""","""reserva""","""directo""","""recibo""",-1,1,2023-01-15,2023-07-15,2023-07-15,2025-01-15,1200.0
"""garantia_extendida_en_riesgo""","""produccion_directo""","""directo""","""920350354""",2024-07-01,"""9024605326""",9024605326,"""DANOS""","""DANOS""","""920583545""","""01""","""009""","""sucursal""","""plan_2""","""emision""","""COP""",2024-07-01,2023-01-15,2025-01-15,2023-01-15,2025-01-15,1200.0,"""ifrs17_corporativo""","""prima""","""reserva""","""directo""","""cobertura""",-1,0,2023-01-15,2024-07-01,2024-07-01,2025-01-15,1200.0
"""garantia_extendida_en_riesgo""","""produccion_directo""","""directo""","""920350354""",2024-07-01,"""9024605326""",9024605326,"""DANOS""","""DANOS""","""920583545""","""01""","""009""","""sucursal""","""plan_2""","""emision""","""COP""",2024-07-01,2023-01-15,2025-01-15,2023-01-15,2025-01-15,1200.0,"""ifrs17_local""","""prima""","""reserva""","""directo""","""cobertura""",-1,0,2023-01-15,2024-07-01,2024-07-01,2025-01-15,1200.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""decenal_financiacion""","""produccion_directo""","""directo""","""003000000545""",2024-12-15,"""003000000723""",3000000723,"""RECONSTRUCCION""","""RECONSTRUCCION""","""00300000072301""","""01""","""003""","""sucursal""","""VIVIENDA_SEGURA""","""emision""","""COP""",2024-12-15,2024-12-15,2034-12-15,2024-12-15,2034-12-15,3.1545619e8,"""ifrs4""","""prima""","""reserva""","""directo""","""recibo""",-1,1,2024-12-15,2024-12-15,2024-12-15,2034-12-15,3.1545619e8
"""matematica_local_directo""","""matematica_local_directo""","""directo""","""190358223""",2024-01-30,"""5920382423""",5920382423,"""SCP""","""SCP""","""259433432""","""02""","""095""","""sucursal""","""plan_1""","""emision""","""COP""",2024-01-30,2024-01-30,2025-01-30,2024-01-30,2025-01-30,1200.0,null,null,null,null,null,null,1,null,2024-01-30,2024-01-30,null,1200.0
"""mensual_pre_vigencia""","""produccion_directo""","""directo""","""634535346""",2023-12-15,"""2938431131""",2938431131,"""AMBULATORIO""","""AMBULATORIO""","""634535766""","""02""","""091""","""sucursal""",

In [6]:
# Se añade el paso de anexar info financiacion
input_pdn_final = prep_data.anexar_info_financiacion(
    input_prod_dir,
    param_compfin,
    df_ipc,
    df_factores_interes,
    FECHA_VALORACION    
)
input_pdn_final.filter(pl.col('aplica_comp_financ') == 1)

aux_casuistica,tipo_insumo,tipo_negocio,poliza,fecha_expedicion_poliza,recibo,numero_documento_sap,amparo,cdsubgarantia,poliza_certificado,compania,ramo_sura,canal,producto,tipo_op,moneda,fecha_contabilizacion_recibo,fecha_inicio_vigencia_recibo,fecha_fin_vigencia_recibo,fecha_inicio_vigencia_cobertura,fecha_fin_vigencia_cobertura,valor_prima_emitida,tipo_contabilidad,componente,clasificacion_adicional,tipo_contrato,nivel_detalle,signo_constitucion,candidato_devengo_50_50,fecha_inicio_vigencia,fecha_constitucion,fecha_inicio_devengo,fecha_fin_devengo,valor_base_devengo,aplica_comp_financ,aplica_ipc_mensual,pais_curva,moneda_curva,meses_max_vigencia,mes_inicio_vigencia,mes_fin_vigencia,mes_valoracion,mes_valoracion_anterior,dias_vig_ini,dias_nodo_ini,dias_vig_fin,dias_nodo_fin,indice_ipc_ini,tasa_ipc_ini,indice_ipc_actual,tasa_ipc_actual,indice_ipc_anterior,tasa_ipc_anterior,fact_acum_val,sum_desc_lir_val,tasa_fwd_real_val,fact_acum_ant,sum_desc_lir_ant,tasa_fwd_real_ant,desc_lir_nodo_ini,fact_acum_ini,sum_desc_lir_nodo_fin,desc_lir_nodo_fin
str,str,str,str,date,str,i64,str,str,str,str,str,str,str,str,str,date,date,date,date,date,f64,str,str,str,str,str,i64,i32,date,date,date,date,f64,i64,i64,str,str,i64,i32,i32,i32,i32,i8,i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""decenal_financiacion""","""produccion_directo""","""directo""","""003000000545""",2024-12-15,"""003000000723""",3000000723,"""RECONSTRUCCION""","""RECONSTRUCCION""","""00300000072301""","""01""","""003""","""sucursal""","""VIVIENDA_SEGURA""","""emision""","""COP""",2024-12-15,2024-12-15,2034-12-15,2024-12-15,2034-12-15,3.1545619e8,"""ifrs17_corporativo""","""prima""","""reserva""","""directo""","""cobertura""",-1,0,2024-12-15,2024-12-15,2024-12-15,2034-12-15,3.1545619e8,1,1,"""CO""","""UVR""",120,202412,203412,202509,202508,17,31,15,31,1.352467,0.004587,1.414035,0.003196,1.40953,0.0018768,1.029176,9.843407,0.00288,1.026221,8.871756,0.00288,0.997128,1.00288,94.78112,0.597977
"""decenal_financiacion""","""produccion_directo""","""directo""","""003000000545""",2024-12-15,"""003000000723""",3000000723,"""RECONSTRUCCION""","""RECONSTRUCCION""","""00300000072301""","""01""","""003""","""sucursal""","""VIVIENDA_SEGURA""","""emision""","""COP""",2024-12-15,2024-12-15,2034-12-15,2024-12-15,2034-12-15,3.1545619e8,"""ifrs17_local""","""prima""","""reserva""","""directo""","""cobertura""",-1,0,2024-12-15,2024-12-15,2024-12-15,2034-12-15,3.1545619e8,1,1,"""CO""","""UVR""",120,202412,203412,202509,202508,17,31,15,31,1.352467,0.004587,1.414035,0.003196,1.40953,0.0018768,1.029176,9.843407,0.00288,1.026221,8.871756,0.00288,0.997128,1.00288,94.78112,0.597977


In [8]:
out_cf_test = devg.devengar(input_pdn_final.filter(pl.col('aplica_comp_financ') == 1),
                            FECHA_VALORACION)
out_cf_test.write_clipboard()
out_cf_test

aux_casuistica,tipo_insumo,tipo_negocio,poliza,fecha_expedicion_poliza,recibo,numero_documento_sap,amparo,cdsubgarantia,poliza_certificado,compania,ramo_sura,canal,producto,tipo_op,moneda,fecha_contabilizacion_recibo,fecha_inicio_vigencia_recibo,fecha_fin_vigencia_recibo,fecha_inicio_vigencia_cobertura,fecha_fin_vigencia_cobertura,valor_prima_emitida,clasificacion_adicional,nivel_detalle,signo_constitucion,candidato_devengo_50_50,fecha_inicio_vigencia,aplica_comp_financ,aplica_ipc_mensual,meses_max_vigencia,mes_inicio_vigencia,mes_fin_vigencia,mes_valoracion,mes_valoracion_anterior,dias_vig_ini,dias_nodo_ini,dias_vig_fin,…,peso_nodo_ini,peso_nodo_fin,peso_nodo_actual,peso_nodo_anterior,ajuste_base_ini,factor_cap_real_ant,sum_desc_valoracion,sum_desc_anterior,suma_factores_remanente_ant,factor_ajuste_ipc_ant,componente,tipo_contabilidad,regla_devengo,tipo_contrato,cohorte,anio_liberacion,transicion,fecha_inicio_periodo,fecha_valoracion,fecha_constitucion,fecha_inicio_devengo,fecha_fin_devengo,valor_base_devengo,pais_curva,moneda_curva,factor_ajuste_ipc,factor_cap_real,suma_factores_remanente,suma_factores_total,saldo_anterior,tasa_acreditacion,acreditacion_intereses,estado_devengo,valor_constitucion,valor_liberacion,valor_liberacion_acum,saldo
str,str,str,str,date,str,i64,str,str,str,str,str,str,str,str,str,date,date,date,date,date,f64,str,str,i64,i32,date,i64,i64,i64,i32,i32,i32,i32,i8,i8,i8,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,i32,str,str,date,date,date,date,date,f64,str,str,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64
"""decenal_financiacion""","""produccion_directo""","""directo""","""003000000545""",2024-12-15,"""003000000723""",3000000723,"""RECONSTRUCCION""","""RECONSTRUCCION""","""00300000072301""","""01""","""003""","""sucursal""","""VIVIENDA_SEGURA""","""emision""","""COP""",2024-12-15,2024-12-15,2034-12-15,2024-12-15,2034-12-15,3.1545619e8,"""reserva""","""cobertura""",-1,0,2024-12-15,1,1,120,202412,203412,202509,202508,17,31,15,…,0.548387,0.483871,1.0,1.0,0.998702,1.024889,9.393091,8.42144,85.600731,1.04481,"""prima""","""ifrs17_corporativo""","""componente_financiacion""","""directo""",2024,"""anio_anterior""","""0""",2025-09-01,2025-09-30,2024-12-15,2024-12-15,2034-12-15,3.1545619e8,"""CO""","""UVR""",1.04815,1.027841,84.629081,94.022171,-3.0754e8,0.006086,-1.8715e6,"""en_curso""",-0.0,3.5121e6,null,-3.0590e8
"""decenal_financiacion""","""produccion_directo""","""directo""","""003000000545""",2024-12-15,"""003000000723""",3000000723,"""RECONSTRUCCION""","""RECONSTRUCCION""","""00300000072301""","""01""","""003""","""sucursal""","""VIVIENDA_SEGURA""","""emision""","""COP""",2024-12-15,2024-12-15,2034-12-15,2024-12-15,2034-12-15,3.1545619e8,"""reserva""","""cobertura""",-1,0,2024-12-15,1,1,120,202412,203412,202509,202508,17,31,15,…,0.548387,0.483871,1.0,1.0,0.998702,1.024889,9.393091,8.42144,85.600731,1.04481,"""prima""","""ifrs17_local""","""componente_financiacion""","""directo""",2024,"""anio_anterior""","""0""",2025-09-01,2025-09-30,2024-12-15,2024-12-15,2034-12-15,3.1545619e8,"""CO""","""UVR""",1.04815,1.027841,84.629081,94.022171,-3.0754e8,0.006086,-1.8715e6,"""en_curso""",-0.0,3.5121e6,null,-3.0590e8
